# 02 — Clustering: Execução Individual de Algoritmos

**Objetivo**: executar cada algoritmo de clustering disponível no projeto
(K-Means, DBSCAN, Aglomerativo, Mean Shift) individualmente, avaliar os resultados
com métricas internas e visualizar os clusters formados.

## Tópicos
1. Pré-processamento: escalonamento com `StandardScaler`
2. K-Means — escolha de `k`, inércia, silhouette
3. DBSCAN — `eps`, `min_samples`, detecção de ruído
4. Clustering Aglomerativo — `linkage`, dendrograma
5. Mean Shift — estimativa de `bandwidth`, clusters automáticos
6. Avaliação: Silhouette Score, Davies-Bouldin Index, Calinski-Harabász Score
7. Visualização dos clusters em PCA 2D

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# ml-clustering-lab runners (implementações futuras)
# from ml_clustering_lab.clustering import get_algorithm

%matplotlib inline
sns.set_theme(style='whitegrid')

# Carrega e pré-processa o dataset
iris = load_iris(as_frame=True)
X = iris.data.values
y_true = iris.target.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Redução para visualização 2D
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)
print(f'Dataset: {X_scaled.shape[0]} amostras, {X_scaled.shape[1]} features')

## 2. K-Means

In [ ]:
from sklearn.cluster import KMeans

# Quando KMeansRunner estiver implementado:
# from ml_clustering_lab.clustering import get_algorithm
# km = get_algorithm('kmeans', n_clusters=3)
# labels_km = km.fit_predict(X_scaled)

km = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels_km = km.fit_predict(X_scaled)
print(f'Inércia: {km.inertia_:.2f}')
print(f'Silhouette: {silhouette_score(X_scaled, labels_km):.4f}')

## 3. DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.5, min_samples=5)
labels_db = db.fit_predict(X_scaled)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = (labels_db == -1).sum()
print(f'Clusters encontrados: {n_clusters_db}')
print(f'Pontos de ruído: {n_noise_db}')
if n_clusters_db > 1:
    mask = labels_db != -1
    print(f'Silhouette (sem ruído): {silhouette_score(X_scaled[mask], labels_db[mask]):.4f}')

## 4. Clustering Aglomerativo

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg.fit_predict(X_scaled)
print(f'Silhouette: {silhouette_score(X_scaled, labels_agg):.4f}')

# Dendrograma (scipy)
Z = linkage(X_scaled, method='ward')
plt.figure(figsize=(12, 4))
dendrogram(Z, truncate_mode='level', p=5, leaf_rotation=90)
plt.title('Dendrograma — Clustering Aglomerativo (ward)')
plt.tight_layout()
plt.show()

## 5. Mean Shift

In [ ]:
from sklearn.cluster import MeanShift, estimate_bandwidth

bw = estimate_bandwidth(X_scaled, quantile=0.2, n_samples=100, random_state=42)
ms = MeanShift(bandwidth=bw, bin_seeding=True)
labels_ms = ms.fit_predict(X_scaled)
n_clusters_ms = len(set(labels_ms))
print(f'Bandwidth estimado: {bw:.4f}')
print(f'Clusters encontrados: {n_clusters_ms}')
if n_clusters_ms > 1:
    print(f'Silhouette: {silhouette_score(X_scaled, labels_ms):.4f}')

## 7. Visualização em PCA 2D

In [ ]:
algo_labels = [
    ('K-Means', labels_km),
    ('DBSCAN', labels_db),
    ('Aglomerativo', labels_agg),
    ('Mean Shift', labels_ms),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (title, labels) in zip(axes, algo_labels):
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab10', s=30, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax)
plt.suptitle('Clusters em PCA 2D — Iris', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()